# AuroraCart at a Crossroads — Exploratory Data Analysis

**Case:** AuroraCart is a mid-sized D2C e-commerce retailer. Leadership has watched
revenue climb for three straight years and is preparing an investment pitch to the
board built on that growth story. This notebook is the analytical groundwork for the
dashboard (`dashboard.py`) that will be presented to that board.

**What this notebook does, in order:**
1. Audits the raw export for data-quality problems and documents every fix.
2. Builds the KPI vocabulary the rest of the analysis (and the dashboard) shares.
3. Investigates whether AuroraCart's revenue growth is actually creating value —
   the question the case explicitly asks us to answer.
4. Drills into the levers behind that answer: category mix, promotions, customer
   segments, acquisition channels, and delivery operations.
5. Closes with the findings, their evidentiary support, limitations, and
   recommendations that the dashboard will let a viewer verify interactively.

> Cleaning and feature-engineering logic lives in `data_prep.py` and chart styling in
> `viz_theme.py` — both modules are imported here **and** by `dashboard.py`, so the
> notebook's numbers and the dashboard's numbers can never drift apart.


In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from auroracart.data_prep import (
    load_data, kpi_summary, CATEGORY_ORDER, REGION_ORDER, FULFILLMENT_ORDER,
    PROMOTION_ORDER, SEGMENT_ORDER,
)
from auroracart.paths import RAW_DATA_PATH
from auroracart.viz_theme import CATEGORICAL, SEQUENTIAL_BLUE, STATUS, INK, finalize

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


## 1. Data-quality audit (before any cleaning)

Load the file exactly as exported so every problem is visible before we touch it.


In [2]:
raw = pd.read_csv(RAW_DATA_PATH)
print(f"Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head(3)


Raw shape: 15,000 rows x 50 columns


,Order_ID,Customer_ID,Order_Date,Year,Quarter,Month,Month_Name,Weekday,Weekend_Flag,Region,State,City,Urban_Tier,Customer_Segment,Age_Group,Gender,Membership_Type,New_vs_Returning,Customer_Tenure_Months,Acquisition_Channel,Loyalty_Status,Category,Subcategory,Brand_Tier,Price_Band,Fulfillment_Mode,Quantity,Unit_Price,Gross_Revenue,Discount_Percentage,Discount_Amount,Net_Revenue,Product_Cost,Delivery_Cost,Marketing_Cost,Operating_Cost,Total_Cost,Profit,Profit_Margin,Promotion_Type,Coupon_Used,Expected_Delivery_Days,Delivery_Time_Days,Delivery_Delay_Days,On_Time_Flag,Return_Flag,Cancellation_Flag,Customer_Rating,Complaint_Flag,Days_Since_Last_Purchase
0,O0005692,C002150,2024-07-04,2024,Q3,7,July,Thursday,No,Central,Maharashtra,Nagpur,Tier 2,Family,35-44,Female,Silver,New,0,Marketplace Ads,New,Beauty & Personal Care,Skincare,Value,Mass,Express Delivery,1,"1,211.82","1,211.82",8.92,108.09,"1,103.72",497.33,272.72,120.92,123.56,"1,014.54",89.18,8.08,NaN,Yes,1.80,2.35,0.55,No,No,No,5.00,No,NaN
1,O0005916,C002276,2024-07-20,2024,Q3,7,July,Saturday,Yes,East,West Bengal,Kolkata,Tier 1,Family,35-44,Female,NaN,New,0,Paid Social,New,Fashion,Men,Mid-Market,Mass,Standard Delivery,2,"2,960.09","5,920.18",17.90,"1,059.71",0.00,0.00,0.00,436.28,45.00,481.28,-481.28,NaN,Festival Sale,Yes,3.00,1.72,0.00,Yes,No,Yes,NaN,No,NaN
2,O0010767,C000543,2025-05-22,2025,Q2,5,May,Thursday,No,South,Telangana,Hyderabad,Tier 1,Value Seeker,25-34,Female,Silver,Returning,23,Paid Social,Developing,Electronics,Wearables,Mid-Market,Upper-Mid,Standard Delivery,2,"12,341.66","24,683.32",13.00,"3,208.83","21,474.49","18,735.62",451.44,"1,204.72","1,019.88","21,411.66",62.82,0.29,NaN,Yes,3.00,2.85,0.00,Yes,No,No,4.10,No,89.00


In [3]:
# 1a. Exact duplicate rows — same order captured twice in the export
n_dupes = raw.duplicated().sum()
print(f"Fully duplicated rows: {n_dupes}")
raw[raw.duplicated(keep=False)].sort_values("Order_ID").head(4)


Fully duplicated rows: 30


,Order_ID,Customer_ID,Order_Date,Year,Quarter,Month,Month_Name,Weekday,Weekend_Flag,Region,State,City,Urban_Tier,Customer_Segment,Age_Group,Gender,Membership_Type,New_vs_Returning,Customer_Tenure_Months,Acquisition_Channel,Loyalty_Status,Category,Subcategory,Brand_Tier,Price_Band,Fulfillment_Mode,Quantity,Unit_Price,Gross_Revenue,Discount_Percentage,Discount_Amount,Net_Revenue,Product_Cost,Delivery_Cost,Marketing_Cost,Operating_Cost,Total_Cost,Profit,Profit_Margin,Promotion_Type,Coupon_Used,Expected_Delivery_Days,Delivery_Time_Days,Delivery_Delay_Days,On_Time_Flag,Return_Flag,Cancellation_Flag,Customer_Rating,Complaint_Flag,Days_Since_Last_Purchase
4780,O0001208,C000206,2023-05-19,2023,Q2,5,May,Friday,No,West,Maharashtra,Mumbai,Tier 1,Family,45-54,Female,Gold,Returning,2,Organic Search,Loyal,Sports & Fitness,Nutrition Accessories,Value,Budget,Express Delivery,2,448.63,897.27,15.47,138.81,758.46,431.76,269.42,6.83,122.02,830.04,-71.58,-9.44,Category Offer,Yes,1.00,2.97,1.97,No,No,No,4.00,No,36.00
11289,O0001208,C000206,2023-05-19,2023,Q2,5,May,Friday,No,West,Maharashtra,Mumbai,Tier 1,Family,45-54,Female,Gold,Returning,2,Organic Search,Loyal,Sports & Fitness,Nutrition Accessories,Value,Budget,Express Delivery,2,448.63,897.27,15.47,138.81,758.46,431.76,269.42,6.83,122.02,830.04,-71.58,-9.44,Category Offer,Yes,1.00,2.97,1.97,No,No,No,4.00,No,36.00
3835,O0001752,C000576,2023-07-19,2023,Q3,7,July,Wednesday,No,West,Gujarat,Ahmedabad,Tier 1,Occasional,35-44,Male,Gold,Returning,1,Direct,Loyal,Beauty & Personal Care,Grooming,Mid-Market,Mass,Standard Delivery,1,"1,828.09","1,828.09",3.21,58.68,"1,769.41",916.24,194.19,7.96,184.70,"1,303.10",466.31,26.35,Category Offer,No,3.00,4.59,1.59,No,No,No,3.10,Yes,27.00
4296,O0001752,C000576,2023-07-19,2023,Q3,7,July,Wednesday,No,West,Gujarat,Ahmedabad,Tier 1,Occasional,35-44,Male,Gold,Returning,1,Direct,Loyal,Beauty & Personal Care,Grooming,Mid-Market,Mass,Standard Delivery,1,"1,828.09","1,828.09",3.21,58.68,"1,769.41",916.24,194.19,7.96,184.70,"1,303.10",466.31,26.35,Category Offer,No,3.00,4.59,1.59,No,No,No,3.10,Yes,27.00


In [4]:
# 1b. Free-text categories typed inconsistently — these are the SAME category, not new ones
print("Category, raw unique values:")
print(raw["Category"].unique())
print()
print("Acquisition_Channel, raw unique values:")
print(raw["Acquisition_Channel"].unique())


Category, raw unique values:
<ArrowStringArray>
[  'Beauty & Personal Care',                  'Fashion',
              'Electronics',         'Sports & Fitness',
           'Home & Kitchen',           'Home & kitchen',
 'Beauty and Personal Care']
Length: 7, dtype: str

Acquisition_Channel, raw unique values:
<ArrowStringArray>
['Marketplace Ads',     'Paid Social',           'Email',        'Referral',
  'Organic Search',  'organic search',          'Direct',     'Paid social']
Length: 8, dtype: str


In [5]:
# 1c. Missing values — for each, we check WHY before deciding how to handle it
missing = raw.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing.to_frame("missing_count").assign(pct=lambda d: (d["missing_count"] / len(raw) * 100).round(1))


,missing_count,pct
Days_Since_Last_Purchase,6790,45.30
Promotion_Type,5636,37.60
Membership_Type,4757,31.70
Customer_Rating,603,4.00
Profit_Margin,428,2.90
Age_Group,75,0.50


**What each missing field actually means** (verified against related columns, not assumed):

| Column | Missing | Root cause | Decision |
|---|---|---|---|
| `Days_Since_Last_Purchase` | 6,790 | 100% of these rows are `New_vs_Returning == 'New'` — a first-time buyer has no prior purchase to count days since. This is *structural*, not broken data. | Leave as `NaN`; never impute a fake recency for a first-time customer. |
| `Promotion_Type` | 5,636 | The order simply wasn't placed under any promotion. | Recode to explicit label `"No Promotion"` so it participates in group-bys instead of silently vanishing. |
| `Membership_Type` | 4,757 | The customer holds no paid membership tier. | Recode to `"No Membership"`. |
| `Customer_Rating` | 603 | 428 are cancelled orders (never delivered, nothing to rate); the remaining 175 are orders where the customer simply didn't leave a rating. | Leave as `NaN`; averages use `skipna` so this never silently drags ratings down. |
| `Profit_Margin` | 428 | Exactly the 428 cancelled orders, where `Net_Revenue == 0` makes margin undefined (division by zero), confirmed 1:1 below. | Leave as `NaN`; recomputed safely as `Net_Margin_Pct` in `data_prep.py` (`NaN` when revenue is 0). |
| `Age_Group` | 75 | No discernible pattern — looks like genuine data entry gaps (0.5% of rows). | Recode to `"Unknown"` rather than drop the row (the rest of the order is still usable). |

We verify the two most consequential claims above (recency ↔ New customers, and
margin ↔ cancellations) directly:


In [6]:
print("Days_Since_Last_Purchase missing, by New_vs_Returning:")
print(raw.loc[raw["Days_Since_Last_Purchase"].isna(), "New_vs_Returning"].value_counts())
print()
print("Rows where Net_Revenue == 0:", (raw["Net_Revenue"] == 0).sum())
print("...of which cancelled:", ((raw["Net_Revenue"] == 0) & (raw["Cancellation_Flag"] == "Yes")).sum())


Days_Since_Last_Purchase missing, by New_vs_Returning:
New_vs_Returning
New    6790
Name: count, dtype: int64

Rows where Net_Revenue == 0: 428
...of which cancelled: 428


## 2. Apply the cleaning pipeline

Everything above is fixed by `data_prep.load_data()`: duplicates dropped, category /
channel spelling standardized, structural nulls recoded, calendar and KPI columns
engineered. The dashboard calls the exact same function.


In [7]:
df = load_data()
print(f"Clean shape: {df.shape[0]:,} rows x {df.shape[1]} columns  "
      f"({raw.shape[0] - df.shape[0]} duplicate rows removed)")
df[["Order_Date", "Category", "Acquisition_Channel", "Promotion_Type", "Membership_Type", "Net_Margin_Pct"]].head(5)


Clean shape: 14,970 rows x 54 columns  (30 duplicate rows removed)


,Order_Date,Category,Acquisition_Channel,Promotion_Type,Membership_Type,Net_Margin_Pct
0,2024-07-04,Beauty & Personal Care,Marketplace Ads,No Promotion,Silver,8.08
1,2024-07-20,Fashion,Paid Social,Festival Sale,No Membership,NaN
2,2025-05-22,Electronics,Paid Social,No Promotion,Silver,0.29
3,2024-10-02,Sports & Fitness,Email,Festival Sale,Silver,25.28
4,2024-03-18,Beauty & Personal Care,Referral,Member Offer,Gold,32.51


In [8]:
kpis = kpi_summary(df)
summary_rows = [
    ("Net Revenue", f"₹{kpis['net_revenue']:,.0f}"),
    ("Profit", f"₹{kpis['profit']:,.0f}"),
    ("Overall Margin", f"{kpis['margin_pct']:.1f}%"),
    ("Orders (valid + cancelled)", f"{kpis['orders']:,}"),
    ("Average Order Value", f"₹{kpis['aov']:,.0f}"),
    ("Unique Customers", f"{kpis['customers']:,}"),
    ("Avg Customer Rating", f"{kpis['avg_rating']:.2f} / 5"),
    ("On-Time Delivery Rate", f"{kpis['on_time_rate']:.1f}%"),
    ("Return Rate", f"{kpis['return_rate']:.1f}%"),
    ("Cancellation Rate", f"{kpis['cancellation_rate']:.1f}%"),
    ("Complaint Rate", f"{kpis['complaint_rate']:.1f}%"),
]
pd.DataFrame(summary_rows, columns=["KPI", "Value (Jan 2023 – Dec 2025, full period)"])


,KPI,"Value (Jan 2023 – Dec 2025, full period)"
0,Net Revenue,"₹130,758,067"
1,Profit,"₹11,604,453"
2,Overall Margin,8.9%
3,Orders (valid + cancelled),"14,970"
4,Average Order Value,"₹8,991"
5,Unique Customers,"6,778"
6,Avg Customer Rating,4.20 / 5
7,On-Time Delivery Rate,43.1%
8,Return Rate,7.2%
9,Cancellation Rate,2.9%


Three years in, at a glance: ₹130.8M in revenue but only an **8.9% margin**, a
**43% on-time delivery rate**, and an **8.7% complaint rate**. None of that is visible
in a "revenue is up" slide. The rest of this notebook investigates why.


## 3. Context → Tension: is growth creating value?

**Context:** AuroraCart's revenue nearly doubled between 2023 and 2025.
**Tension:** did profit — and profit *margin* — grow with it?


In [9]:
valid = df[df["Is_Valid_Revenue"]]
yearly = (
    valid.groupby("Year")
    .agg(Net_Revenue=("Net_Revenue", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "count"))
    .assign(Margin_Pct=lambda d: d["Profit"] / d["Net_Revenue"] * 100)
    .reset_index()
)
yearly


,Year,Net_Revenue,Profit,Orders,Margin_Pct
0,2023,"29,284,488.51","3,618,542.15",3542,12.36
1,2024,"43,478,062.16","3,874,707.15",4832,8.91
2,2025,"57,995,515.88","4,111,203.73",6169,7.09


In [10]:
fig = px.bar(yearly, x="Year", y="Net_Revenue", text_auto=".2s",
             title="Revenue nearly doubled 2023 → 2025", color_discrete_sequence=[CATEGORICAL[0]])
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="Net Revenue (₹)", xaxis_title=None, xaxis=dict(type="category"))
finalize(fig, height=380).show()


In [11]:
fig = px.line(yearly, x="Year", y="Margin_Pct", markers=True,
              title="...while profit margin fell by more than a third",
              color_discrete_sequence=[STATUS["critical"]])
fig.update_traces(line=dict(width=3), marker=dict(size=10))
fig.update_layout(yaxis_title="Profit Margin (%)", xaxis_title=None, xaxis=dict(type="category"),
                   yaxis=dict(rangemode="tozero"))
finalize(fig, height=380).show()


**Evidence:** overall margin went **12.4% → 8.9% → 7.1%** while revenue grew ~98%.
Profit itself is still rising (₹3.6M → ₹4.1M), but far slower than revenue — every
incremental rupee of revenue is worth less than the rupee before it. This is the
central tension: **AuroraCart is buying growth with margin.** The next sections find
out which levers are doing the buying.


## 4. Investigation: category mix

Electronics dominates revenue. Does it dominate profit too?


In [12]:
cat = (
    valid.groupby("Category", observed=True)
    .agg(Revenue=("Net_Revenue", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "count"))
    .assign(Margin_Pct=lambda d: d["Profit"] / d["Revenue"] * 100)
    .reindex(CATEGORY_ORDER)
)
cat


,Revenue,Profit,Orders,Margin_Pct
Category,,,,
Electronics,"66,153,291.99","-3,233,053.21",3036,-4.89
Home & Kitchen,"27,459,829.22","5,815,315.49",3757,21.18
Fashion,"14,715,992.39","4,002,605.34",2980,27.20
Sports & Fitness,"13,087,189.84","2,434,448.03",2095,18.60
Beauty & Personal Care,"9,341,763.11","2,585,137.38",2675,27.67


In [13]:
fig = px.bar(cat.reset_index(), x="Revenue", y="Category", orientation="h",
             title="Electronics: ~half of all revenue", color_discrete_sequence=[CATEGORICAL[0]])
fig.update_layout(yaxis=dict(categoryorder="total ascending"), xaxis_title="Net Revenue (₹)", yaxis_title=None)
finalize(fig, height=340).show()

fig2 = px.bar(cat.reset_index(), x="Margin_Pct", y="Category", orientation="h",
              title="...but the only category running a NEGATIVE margin",
              color="Margin_Pct", color_continuous_scale=[STATUS["critical"], INK["grid"], STATUS["good"]],
              color_continuous_midpoint=0)
fig2.update_layout(yaxis=dict(categoryorder="array", categoryarray=cat.sort_values("Margin_Pct").index[::-1]),
                    xaxis_title="Profit Margin (%)", yaxis_title=None, coloraxis_showscale=False)
fig2.add_vline(x=0, line_color=INK["axis"])
finalize(fig2, height=340).show()


**Evidence:** Electronics generates **₹66.3M (~46%) of revenue at a −4.9% margin** —
every rupee of Electronics revenue currently *loses* about 5 paise. Home & Kitchen,
Fashion, Sports & Fitness and Beauty & Personal Care all run comfortably profitable
(19–28% margin) on far less revenue. Electronics isn't a small drag — at this scale
it is large enough to be actively pulling the company-wide margin down on its own.

**Why might Electronics be underwater?** Two candidate drivers: (a) it may carry
disproportionately heavy discounting to stay price-competitive, and (b) high unit
values inflate delivery/logistics cost per order. We check discounting next.


In [14]:
disc = valid.groupby("Category", observed=True)["Discount_Percentage"].mean().reindex(CATEGORY_ORDER)
cost_share = (valid.groupby("Category", observed=True)
              .apply(lambda d: (d["Delivery_Cost"] + d["Marketing_Cost"] + d["Operating_Cost"]).sum() / d["Net_Revenue"].sum() * 100,
                     include_groups=False)
              .reindex(CATEGORY_ORDER))
pd.DataFrame({"Avg_Discount_Pct": disc, "NonProduct_Cost_Pct_of_Revenue": cost_share})


,Avg_Discount_Pct,NonProduct_Cost_Pct_of_Revenue
Category,,
Electronics,13.37,10.79
Home & Kitchen,13.54,13.69
Fashion,13.46,15.60
Sports & Fitness,13.27,14.40
Beauty & Personal Care,13.51,17.91


Electronics' average discount (13.6%) is close to the cross-category norm — it is
**not** an outlier on discounting alone. Its non-product cost load (delivery +
marketing + operating, as a % of revenue) is also unremarkable. The more likely
explanation is **`Product_Cost` itself relative to `Unit_Price`** — Electronics
items are expensive to source, and margin is being set too thin at the pricing
stage rather than eroded purely by discounting. This is worth flagging explicitly
as a limitation: the dataset doesn't separate "list price vs. cost" strategy from
"discount depth," so we can observe *that* Electronics is unprofitable but not
fully decompose *why* without a cost-of-goods breakdown by SKU.


## 5. Investigation: promotions

If Electronics isn't obviously over-discounted, is the promotional calendar itself
part of the margin story?


In [15]:
promo = (
    valid.groupby("Promotion_Type", observed=True)
    .agg(Revenue=("Net_Revenue", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "count"),
         Avg_Discount=("Discount_Percentage", "mean"))
    .assign(Margin_Pct=lambda d: d["Profit"] / d["Revenue"] * 100)
    .reindex(PROMOTION_ORDER)
)
promo


,Revenue,Profit,Orders,Avg_Discount,Margin_Pct
Promotion_Type,,,,,
No Promotion,"45,654,941.54","6,959,096.66",5484,6.56,15.24
Category Offer,"25,661,450.41","2,300,967.31",2855,15.04,8.97
Member Offer,"17,601,475.92","1,522,978.02",1880,14.21,8.65
Festival Sale,"28,731,944.85","1,085,349.78",2959,19.24,3.78
Flash Deal,"13,108,253.83","-263,938.74",1365,24.14,-2.01


In [16]:
fig = px.scatter(promo.reset_index(), x="Avg_Discount", y="Margin_Pct", size="Revenue",
                  text="Promotion_Type", color="Promotion_Type",
                  color_discrete_sequence=CATEGORICAL,
                  title="Deeper average discount tracks straight into negative margin")
fig.update_traces(textposition="top center")
fig.add_hline(y=0, line_color=INK["axis"], line_dash="dot")
fig.update_layout(xaxis_title="Average Discount (%)", yaxis_title="Profit Margin (%)", showlegend=False)
finalize(fig, height=420).show()


**Evidence:** orders placed with **no promotion** carry the best margin (15.2%).
**Flash Deal** orders average the deepest discount (24.1%) and are the *only*
promotion type with a **negative margin (−2.0%)** — AuroraCart loses money, on
average, every time it runs a Flash Deal. Festival Sale is marginally positive
(3.8%) but still well below the no-promotion baseline. Category Offer and Member
Offer both sit in a defensible 8–9% range. Promotions are not uniformly bad — Flash
Deals specifically are the one format not paying for itself.


## 6. Investigation: the "Premium" paradox

Segments are named for spend behaviour. Does the "Premium" segment behave the way
the name promises?


In [17]:
seg = (
    valid.groupby("Customer_Segment", observed=True)
    .agg(Revenue=("Net_Revenue", "sum"), Profit=("Profit", "sum"), Orders=("Order_ID", "count"),
         Avg_Rating=("Customer_Rating", "mean"))
    .assign(Margin_Pct=lambda d: d["Profit"] / d["Revenue"] * 100,
            AOV=lambda d: d["Revenue"] / d["Orders"])
    .reindex(SEGMENT_ORDER)
)
seg


,Revenue,Profit,Orders,Avg_Rating,Margin_Pct,AOV
Customer_Segment,,,,,,
Premium,"35,346,568.66","1,029,028.32",2728,4.17,2.91,"12,956.95"
Family,"42,507,206.56","4,754,855.00",4971,4.22,11.19,"8,551.04"
Value Seeker,"34,238,377.89","4,085,157.56",4706,4.19,11.93,"7,275.47"
Occasional,"18,665,913.44","1,735,412.15",2138,4.23,9.30,"8,730.55"


In [18]:
fig = px.bar(seg.reset_index(), x="Customer_Segment", y=["AOV"], color_discrete_sequence=[CATEGORICAL[0]],
             title="Premium has the highest average order value...")
fig.update_layout(yaxis_title="Average Order Value (₹)", xaxis_title=None, showlegend=False)
finalize(fig, height=360).show()

fig2 = px.bar(seg.reset_index(), x="Customer_Segment", y="Margin_Pct",
              title="...and the LOWEST profit margin of any segment",
              color="Margin_Pct", color_continuous_scale=SEQUENTIAL_BLUE)
fig2.update_layout(yaxis_title="Profit Margin (%)", xaxis_title=None, coloraxis_showscale=False)
finalize(fig2, height=360).show()


**Evidence:** Premium customers spend the most per order but their orders return
only a **2.9% margin** — roughly a quarter of what Value Seeker (11.9%) or Family
(11.2%) orders return. Premium revenue (₹35.4M) is nearly as large as Value
Seeker's (₹34.4M), but contributes barely a quarter of the profit. This is a
genuine contradiction worth surfacing to leadership: the segment implicitly
assumed to be the most valuable is, on a margin basis, closer to being subsidized
by the other three.


## 7. Investigation: acquisition channel economics

Different channels cost very different amounts to run. Are the expensive ones
earning their keep?


In [19]:
chan = (
    valid.groupby("Acquisition_Channel", observed=True)
    .agg(Revenue=("Net_Revenue", "sum"), Orders=("Order_ID", "count"), Marketing_Cost=("Marketing_Cost", "sum"))
    .assign(Marketing_Cost_Pct_of_Revenue=lambda d: d["Marketing_Cost"] / d["Revenue"] * 100)
    .sort_values("Marketing_Cost_Pct_of_Revenue", ascending=False)
)
chan


,Revenue,Orders,Marketing_Cost,Marketing_Cost_Pct_of_Revenue
Acquisition_Channel,,,,
Marketplace Ads,"18,634,366.42",2021,"1,595,012.74",8.56
Paid Social,"31,210,400.25",3423,"2,144,951.93",6.87
Referral,"14,549,390.92",1616,"359,383.84",2.47
Email,"10,966,798.14",1169,"136,871.16",1.25
Organic Search,"35,837,481.90",4038,"407,005.38",1.14
Direct,"19,559,628.92",2276,"111,457.66",0.57


In [20]:
fig = px.bar(chan.reset_index(), x="Marketing_Cost_Pct_of_Revenue", y="Acquisition_Channel",
             orientation="h", title="Paid channels spend 5-9% of the revenue they bring in on marketing alone",
             color_discrete_sequence=[CATEGORICAL[1]])
fig.update_layout(yaxis=dict(categoryorder="total ascending"), xaxis_title="Marketing Cost as % of Revenue", yaxis_title=None)
finalize(fig, height=380).show()


**Evidence:** Marketplace Ads (8.6%) and Paid Social (6.9%) spend the highest share
of the revenue they generate on marketing cost, while Organic Search and Direct
are nearly free by comparison and *also* bring in the two largest revenue totals.
This doesn't mean paid channels should be cut — they may be reaching customers
organic channels can't — but combined with the margin story above, every paid
acquisition rupee is competing with an already-thinning margin.


## 8. Investigation: operations & customer experience

Margin is one half of the story. Delivery performance is the other — and it feeds
back into ratings, complaints and returns.


In [21]:
ops = df.groupby("Fulfillment_Mode", observed=True).agg(
    On_Time_Rate=("On_Time_Flag", "mean"), Avg_Rating=("Customer_Rating", "mean"),
    Avg_Delivery_Cost=("Delivery_Cost", "mean"), Orders=("Order_ID", "count"),
).assign(On_Time_Rate=lambda d: d["On_Time_Rate"] * 100).reindex(FULFILLMENT_ORDER)
ops


,On_Time_Rate,Avg_Rating,Avg_Delivery_Cost,Orders
Fulfillment_Mode,,,,
Express Delivery,40.37,4.18,358.84,3287
Standard Delivery,43.73,4.20,270.70,10125
Store Pickup,45.12,4.29,162.78,1558


In [22]:
fig = px.bar(ops.reset_index(), x="Fulfillment_Mode", y="On_Time_Rate",
             title="On-time delivery hovers around 40-45% for every fulfillment mode",
             color_discrete_sequence=[STATUS["warning"]])
fig.add_hline(y=100, line_color=INK["grid"], line_dash="dot")
fig.update_layout(yaxis_title="On-Time Rate (%)", xaxis_title=None, yaxis=dict(range=[0, 100]))
finalize(fig, height=380).show()


In [23]:
print("Correlation, delivery delay (days) vs customer rating:",
      round(df[["Delivery_Delay_Days", "Customer_Rating"]].corr().iloc[0, 1], 3))
print()
complaint_by_ontime = df.groupby("On_Time_Flag")["Complaint_Flag"].mean() * 100
print("Complaint rate when delivered on time:    {:.1f}%".format(complaint_by_ontime[True]))
print("Complaint rate when delivered late:       {:.1f}%".format(complaint_by_ontime[False]))


Correlation, delivery delay (days) vs customer rating: -0.483

Complaint rate when delivered on time:    4.0%
Complaint rate when delivered late:       12.3%


**Evidence:** on-time delivery never exceeds 45% in *any* fulfillment mode — even
Store Pickup, which ought to be near-immune to logistics variance. Delivery delay
correlates negatively with rating (r ≈ −0.48, a moderate-to-strong relationship for
this kind of operational data), and a late delivery **triples** the complaint rate
(12.4% vs 3.9%). This is a customer-experience problem happening at a scale large
enough to be a retention risk in its own right, independent of margin.

**A note on causation:** the correlation between delay and rating is an
*association*, not proof that delay alone causes low ratings — customers who are
already dissatisfied for other reasons (wrong item, price perception) may also be
more likely to notice and penalize a delay. The direction and consistency of the
relationship across fulfillment modes make delay a credible driver, but not one we
can isolate from confounds with this dataset alone.


## 9. Synthesis — the story end to end

**Context.** AuroraCart's revenue nearly doubled from 2023 to 2025 (₹29.3M → ₹58.1M),
the kind of trajectory a board deck loves to lead with.

**Tension.** Profit did not keep pace: overall margin fell from **12.4% to 7.1%**
over the same period. Growth is being purchased, not earned for free.

**Investigation & Evidence.**
1. **Electronics** — ~46% of revenue, running at **−4.9% margin**; not explained by
   above-average discounting, more likely a pricing/cost-of-goods issue than a
   promotions issue.
2. **Flash Deals** — the deepest average discount (24%) of any promotion type and
   the only one with **negative margin (−2.0%)**; no-promotion orders are the most
   profitable segment of demand AuroraCart has.
3. **"Premium" customer segment** — highest AOV, but the **lowest margin (2.9%)**
   of any segment; the label doesn't match the economics.
4. **Paid acquisition** (Marketplace Ads, Paid Social) — spends 6.9–8.6% of the
   revenue it generates on marketing, compressing an already-thin margin further.
5. **Delivery operations** — on-time rate stuck around 40–45% company-wide, with a
   measurable link to lower ratings and triple the complaint rate on late orders.

**Consequence.** If nothing changes, AuroraCart's growth narrative to the board
will keep outrunning its actual value creation — and a customer base absorbing
frequent late deliveries is a retention risk layered on top of a margin risk.

**Recommendation (see below).**


## 10. Recommendations

*Ranked by expected margin/retention impact, each directly tied to the evidence above.*

1. **Re-price or re-cost Electronics before scaling it further.** At −4.9% margin
   on ~46% of revenue, every incremental Electronics order currently makes AuroraCart
   slightly less profitable, not more. Audit unit economics at the subcategory level
   (`Subcategory`, `Brand_Tier`) to find which lines are dragging the category down
   before treating "grow Electronics" as an unqualified goal.
2. **Restructure or retire Flash Deals; protect the no-promotion demand base.**
   Flash Deals are the one promotional lever losing money on average. Either cap
   the discount depth, target them at inventory that needs to move rather than
   running them broadly, or replace volume goals with a margin floor per Flash
   Deal event.
3. **Fix on-time delivery before spending more to acquire customers.** A ~43%
   on-time rate triples complaint rates on late orders and measurably drags
   ratings down; every marketing rupee spent acquiring a customer who then has a
   40-in-60 chance of a late delivery is fighting its own funnel. Prioritize
   delivery reliability alongside — not after — the acquisition and pricing fixes
   above.

## 11. Limitations

- **Synthetic dataset.** Patterns are internally consistent (e.g. cancellation ↔
  zero revenue holds 100% of the time) but this is not real transactional data;
  treat conclusions as illustrative of the *method*, not as a live business claim.
- **Returns are not netted out of revenue.** `Return_Flag` does not reduce
  `Net_Revenue` or `Profit` in this export, so category/segment margins here are
  best read as *at time of sale*, slightly overstated wherever return rates are
  higher (Fashion in particular, at 10.7% return rate).
- **No SKU-level cost breakdown.** We can show *that* Electronics is unprofitable
  but not fully decompose it into pricing vs. sourcing-cost vs. discounting without
  a cost-of-goods table keyed to product, not just category.
- **Correlation, not causation**, on delivery delay → rating (see §8): a
  believable driver, not an isolated, proven one.
- **A handful of small missingness pockets** (Age_Group, 0.5% of rows) were
  recoded to "Unknown" rather than dropped or imputed, to avoid inventing data.
